# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Thu Sep 3, 2026 - Day 4: hypothesis testing, finished (on Zoom)

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">


## Today's plan

Tuesday we got as far as a histogram of two stellar populations. Is that difference there, or is it noise? That's a hypothesis test.

Three ways the test will mislead you:

* too little data hides a difference that is there
* too much data makes a trivial one look enormous
* run it enough times and you get a "significant" result out of nothing

Then, at the end: was a single population ever Gaussian - i.e. were Tuesday's fits entitled to their error bars?

Same rule as always - I'd rather take a wrong turn out loud than lose half the room to silence. Chat, or unmute.


## Where we left off on Day 3

## What we covered on Day 3

* Robust statistics: k-sigma clipping, MAD as a robust scale estimate, the Hogg/Bovy/Lang (2010) mixture model, Theil-Sen and RANSAC
    * the theme: least-squares assumes a lot, and real data breaks those assumptions constantly
* The bootstrap: resample with replacement, recompute the statistic thousands of times, and the spread of that is your error bar
    * no formula needed, no distributional assumption either
* Real data: installed `astroML`, pulled up 252,871 SDSS SEGUE stars (SEGUE: an SDSS spectroscopic survey of Milky Way stars, whose low-resolution spectra give temperature, gravity, [Fe/H] and [alpha/Fe]), and looked at [Fe/H] vs. [alpha/Fe] together
    * two populations showed up as a divot in the histogram

<img src="images/hogg2010_fig4_mixture_fit.png" alt="Hogg, Bovy and Lang 2010 Figure 4: a straight-line fit to 20 points with outliers modelled explicitly by a mixture, the inliers picked out" width="760" style="display:block;margin:0 auto;">


<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

🎤 Before we reload the plot: what were the two populations we found in that histogram, and what physically makes them different?

</div>

Cold-call. Don't let the same two or three people answer every question this week - see run sheet for who's due.

## Quick recap: the same plot as Tuesday

In [ ]:
from astroML.datasets import fetch_sdss_sspp
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

data = fetch_sdss_sspp()
good = (data['FeHErr'] < 0.3) & (data['alphFeErr'] < 0.3) & (data['SNR'] > 20)
d = data[good]
print(f"{len(d)} stars after basic quality cuts")

plt.figure(figsize=(6, 5))
plt.hist2d(d['FeH'], d['alphFe'], bins=100, cmap='viridis')
plt.xlabel('[Fe/H]'); plt.ylabel('[alpha/Fe]')
plt.colorbar(label='N stars')
plt.title('The whole sample, before we split anything')
plt.show()

## Why this plot matters

We're looking at a direct measurement of Galactic archaeology: using the present-day chemistry of stars to reconstruct how the Milky Way disk formed.

The two streaks are two different stellar populations. One is metal-richer and lower-alpha, roughly the thin disk, younger, formed over a longer and more chemically enriched history. The other is metal-poorer and alpha-enhanced, roughly the thick disk, older, formed when the Galaxy had less time to enrich itself with iron before core-collapse supernovae seeded it with alpha elements.

<img src="images/mw_thin_thick_disk_diagram.png" alt="Edge-on schematic of the Milky Way with the thin disk, thick disk, bulge, Galactic centre, halo and the Sun labelled, and the 8 kiloparsec Sun to centre distance marked" width="820" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">NatashaFromEndor, CC BY-SA 4.0, via Wikimedia Commons</div>


## The alpha clock

Notation first. [X/Y] is the log10 of the X-to-Y abundance ratio relative to the Sun: [Fe/H] = 0 is solar iron, -1 is a tenth of solar. One unit of that log is a **dex**, so populations 0.75 dex apart differ by a factor of about 5.6.

[alpha/Fe] works as a rough clock because the alpha elements (O, Mg, Si, Ca, Ti) come promptly from core-collapse supernovae, while iron accumulates more slowly from Type Ia supernovae. So how alpha-enhanced a population is tells us something about how fast it formed its stars.

<img src="images/origin_of_elements.png" alt="Periodic table colour-coded by where each element is made, showing the alpha elements dominated by exploding massive stars and iron with a large contribution from exploding white dwarfs" width="980" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">Cmglee, CC BY-SA 3.0, via Wikimedia Commons, from data by Jennifer Johnson (Ohio State)</div>


## The same split, in a bigger survey

Our SEGUE sample is not the only place this shows up. APOGEE DR17 splits the same plane by height above the disk plane, and the alpha-rich population takes over as you climb.

<img src="images/apogee_mgfe_by_height.jpg" alt="APOGEE DR17 magnesium to iron versus iron abundance, in a grid of panels binned by height above the Galactic plane and galactocentric radius, with the high-magnesium sequence dominating the high-altitude panels" width="1000" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">J. Holtzman / Sloan Digital Sky Survey. Rows are |z| (height above the Galactic mid-plane), columns are R (distance from the Galactic centre), colour is orbital eccentricity.</div>

This is the split we'll use for today's hypothesis test. The physics above is what puts the boundary where it is.


## Selection effects in this catalog

This catalog is **not** a random sample of Milky Way stars. It's built from 10 explicit selection cuts, documented in the catalog's own header: a magnitude range, color cuts, proper-motion limits, a surface-gravity range, temperature-error cuts, and more.

The magnitude cut ($14 < r < 21$) is distance-dependent. It changes which stars we can see as a function of how far away they are, which is a classic Malmquist-type bias: far away, only the intrinsically bright stars make the cut.

The proper-motion cut ($|\mu| < 200$ milliarcseconds per year) pre-selects on kinematics, so we cannot use this same sample to do an unbiased kinematic study without correcting for that cut first.

Any statistic we compute naively from this catalog, something like "what fraction of Milky Way stars are alpha-rich," is really answering a question about stars that pass these ten cuts.

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**That gap between the sample and the population does not announce itself in the data.**

</div>

Keep that in mind for the next twenty minutes. Everything we compute today is a statement about stars that passed these ten cuts, and turning it into a statement about the Milky Way is a separate piece of work that nobody is doing on this slide.

<img src="images/sdss_25m_telescope.jpg" alt="The Sloan 2.5 metre telescope in its roll-off enclosure at Apache Point Observatory with the Sacramento Mountains behind it" width="700" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">Fermilab Visual Media Services / Sloan Digital Sky Survey</div>


## The question

Split the stars by alpha-element abundance into an alpha-poor sample ([alpha/Fe] < 0.1) and an alpha-rich one ([alpha/Fe] > 0.3), i.e. the two populations in the 2D histogram.

Do their metallicities come from different distributions, or could the difference in a histogram just be sampling noise?

Our **null hypothesis** $H_0$ (the statistician's $H_0$, not the Hubble constant) is that the two [Fe/H] samples are drawn from the same underlying distribution. We ask the data to talk us out of it.

I should say that this cut throws away every star with $0.1 < [\alpha/\mathrm{Fe}] < 0.3$, which sharpens the contrast by construction. It's legitimate here, since we cut on $\alpha$ and test [Fe/H], but it is a choice, and per the previous slide it belongs in your write-up.

<img src="images/apogee_alpha_fe_feh.png" alt="Alpha element to metals ratio against iron abundance for the full APOGEE DR17 sample, showing a high-alpha and a low-alpha sequence" width="640" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">The same plane in a bigger, newer survey. R. Beaton / Sloan Digital Sky Survey, APOGEE DR17</div>


In [ ]:
alpha_poor = d[d['alphFe'] < 0.1]   # thin-disk-like
alpha_rich = d[d['alphFe'] > 0.3]   # thick-disk-like
print(f"alpha-poor: {len(alpha_poor)} stars, alpha-rich: {len(alpha_rich)} stars")

Notice the ratio: alpha-rich outnumbers alpha-poor almost four to one, in a Galaxy where the thick disk is the minority. That is the selection function from two slides ago made visible - SEGUE pointed mostly away from the Galactic plane, at faint, distant stars, which is thick-disk and halo territory. The catalogue is not the Galaxy.

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.hist2d(d['FeH'], d['alphFe'], bins=100, cmap='Greys')
plt.axhline(0.1, color='C0', lw=2); plt.axhline(0.3, color='C1', lw=2)
plt.axhspan(0.1, 0.3, color='red', alpha=0.15)
plt.text(-2.4, 0.02, 'alpha-poor: kept', color='C0', fontsize=11)
plt.text(-2.4, 0.19, 'discarded by the cut', color='red', fontsize=11)
plt.text(-2.4, 0.37, 'alpha-rich: kept', color='C1', fontsize=11)
plt.xlabel('[Fe/H]'); plt.ylabel('[alpha/Fe]')
n_cut = ((d['alphFe'] >= 0.1) & (d['alphFe'] <= 0.3)).sum()
plt.title(f'The cut discards {n_cut} of {len(d)} stars ({100*n_cut/len(d):.0f}%)')
plt.show()

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 Post one word in chat right now, before the next plot: **same** or **different**. Based on the split you just saw, do you think the two [Fe/H] distributions come from the same underlying population, or not?

</div>

TA relays a quick tally from chat; GN can't see it while sharing full screen.

In [ ]:
for sample, color, name in [(alpha_poor, 'C0', 'alpha-poor'), (alpha_rich, 'C1', 'alpha-rich')]:
    feh = sample['FeH']
    q25, q50, q75 = np.percentile(feh, [25, 50, 75])
    plt.hist(feh, bins=60, density=True, alpha=0.5, color=color, label=name)
    plt.axvline(q50, color=color, lw=2)
    plt.axvspan(q25, q75, color=color, alpha=0.08)
    print(f"{name:11s}: median={q50:+.2f}, IQR=[{q25:+.2f}, {q75:+.2f}], mean={feh.mean():+.2f}")
plt.xlabel('[Fe/H]'); plt.ylabel('density'); plt.legend()
plt.title('Do these look like the same distribution? (line = median, band = IQR)')
plt.show()

By eye they look obviously different - but "obviously" isn't a number, and I keep asking you to back that kind of judgment up quantitatively.

The printed medians and IQR bands are already most of the story, i.e. Tuesday's robust summaries put to work on real data. The KS test turns that comparison into a single number.


## The Kolmogorov-Smirnov two-sample test

The KS test compares the two samples' **empirical CDFs** (ECDFs) - the data's own stand-in for the CDF we met on Day 2. Sort the sample, and at each sorted [Fe/H] value step the curve up by $1/n$, so what you are plotting is cumulative fraction against [Fe/H].

Then it asks what the largest vertical gap between the two ECDFs is, and how often a gap at least that big turns up when both samples do come from the same distribution.

Let's look at the ECDFs directly before we run the test. The test statistic is a number you can read straight off this plot.


In [ ]:
def ecdf(vals):
    s = np.sort(vals)
    return s, np.arange(1, len(s) + 1) / len(s)

x_poor, y_poor = ecdf(alpha_poor['FeH'])
x_rich, y_rich = ecdf(alpha_rich['FeH'])

plt.plot(x_poor, y_poor, label='alpha-poor ECDF')
plt.plot(x_rich, y_rich, label='alpha-rich ECDF')

# find and mark the largest gap, on a common grid
grid = np.linspace(min(x_poor.min(), x_rich.min()), max(x_poor.max(), x_rich.max()), 2000)
cdf_poor = np.searchsorted(x_poor, grid, side='right') / len(x_poor)
cdf_rich = np.searchsorted(x_rich, grid, side='right') / len(x_rich)
gap = np.abs(cdf_poor - cdf_rich)
i_max = np.argmax(gap)
plt.vlines(grid[i_max], cdf_rich[i_max], cdf_poor[i_max], color='k', linestyle='--',
           label=f'D = {gap[i_max]:.3f}')
plt.xlabel('[Fe/H]'); plt.ylabel('cumulative fraction'); plt.legend()
plt.title('The KS statistic is literally this gap')
plt.show()

## The KS statistic, formally

$$D = \max_x \big| F_1(x) - F_2(x) \big|$$

$F_1$ and $F_2$ are the two samples' empirical CDFs, so $D$ is the largest vertical gap between the two ECDFs we plotted. `scipy.stats.ks_2samp` computes $D$ and the p-value for us, and that plot is what it's doing underneath.


In [ ]:
ks_stat, p_value = stats.ks_2samp(alpha_poor['FeH'], alpha_rich['FeH'])
print(f"KS statistic = {ks_stat:.4f}  (matches the gap marked above)")
print(f"p-value = {p_value:.3e}")
# p prints as 0.000e+00 - the true value underflowed machine precision (unpacked on the p-value slide).

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

🎤 Before the next slide: in your own words, what does `p-value = 0.000` actually mean here? Wrong guesses are useful, say the wrong thing out loud.

</div>

Cold-call a different name than the warm-up. See run sheet.

## What does that p-value mean?

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**If the two samples really were drawn from the same distribution, the p-value is the probability of seeing a gap $D$ at least this large from sampling noise alone.**

</div>

So it is a statement about what the data would look like under an assumption you supplied. Two readings people reach for that it is **not**: the probability that the null hypothesis is true, and the probability that the result is due to chance.

Here it's vanishingly small, so the null is a poor description of what we see. The printout says $p = 0.000$, but a probability can't equal zero - the true value underflowed machine precision, so write $p < 10^{-300}$ and never $p = 0$.

Now watch what happens with much smaller samples of the same two populations.


In [ ]:
rng = np.random.default_rng(20260827)
n_small = 8
sub_poor = rng.choice(alpha_poor['FeH'], n_small, replace=False)
sub_rich = rng.choice(alpha_rich['FeH'], n_small, replace=False)
ks_small_sample, p_small_sample = stats.ks_2samp(sub_poor, sub_rich)
print(f"with only {n_small} stars per sample: KS={ks_small_sample:.3f}, p={p_small_sample:.3f}")
print("Same two populations, which we know differ. This p-value says 'not significant.'")

With only 8 stars per sample the same test that was overwhelmingly significant on the full sample now can't reject the null - i.e. a false negative: the difference is real and the test missed it. That's the first of the three traps. (Rejecting at $p < 0.05$ is a convention, not a law. It means you accept being wrong 1 time in 20 when the null is true.) This is one draw with one seed. Repeat it with different draws of 8 and the test rejects about half the time. On two populations whose medians sit 0.75 dex apart, it is close to a coin flip whether you even see the difference. Let's run exactly that.

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A hypothesis test that fails to reject $H_0$ never means the two things are the same.**

</div>

It can just as easily mean our sample was too small to tell.

Sample size and test choice both belong in your write-up every time you report a p-value, and it's the kind of thing an AI assistant won't flag for you unprompted.


In [ ]:
rng_pow = np.random.default_rng(4570902)
n_rep = 2000
p_small = np.array([stats.ks_2samp(rng_pow.choice(alpha_poor['FeH'], 8, replace=False),
                                   rng_pow.choice(alpha_rich['FeH'], 8, replace=False)).pvalue
                    for _ in range(n_rep)])
frac = (p_small < 0.05).mean()
plt.figure(figsize=(7, 4))
plt.hist(p_small, bins=40, color='C7')
plt.axvline(0.05, color='r', lw=2, label='p = 0.05')
plt.xlabel('p-value from one draw of 8 vs 8'); plt.ylabel(f'count out of {n_rep} repeats')
plt.title(f'Same two populations every time. Only {100*frac:.0f}% of draws reject.')
plt.legend(); plt.show()
print(f"rejects at p<0.05 in {100*frac:.1f}% of {n_rep} draws - the test is close to a coin flip at n=8")

<div style="font-size:1.15em; line-height:1.4; margin:0.6em 0; padding:0.5em 0.8em; border-left:4px solid #444; background:#f2f2f2;">

💬 One line in chat: did the small-sample result surprise you, yes or no? If yes, say why in a few words.

</div>

TA reads out 2-3 responses.

## The second trap: large samples

Our full-sample p-value underflowed to zero with 28,148 alpha-poor stars against 104,427 alpha-rich ones. At those sample sizes, even a 0.01 dex difference in medians would come back below any threshold you like. That's smaller than the measurement errors - a difference nobody would write a paper about. The test answers whether there is any difference at all, but the question you care about is whether the difference is worth anything. At LSST scale (Rubin's ~20 billion objects) "p < 0.05" is close to information-free.

So report the **effect size** alongside $p$. Here $D = 0.61$, i.e. the two ECDFs are separated by 61 percentage points at their widest, which is enormous by anyone's standard.

Small samples hide differences that are there and large samples make trivial ones significant, so the two questions you actually care about are both still open.

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The p-value alone answers neither one: whether a difference exists, or whether it is big enough to matter.**

</div>


In [ ]:
rng_n = np.random.default_rng(4570903)
pool = alpha_poor['FeH']
shift = 0.02   # dex - far smaller than the measurement errors, nobody would care
sizes = np.unique(np.logspace(1, 4.1, 25).astype(int))
p_by_n = []
for n in sizes:
    a = rng_n.choice(pool, n, replace=False)
    b = rng_n.choice(pool, n, replace=False) + shift
    p_by_n.append(stats.ks_2samp(a, b).pvalue)
plt.figure(figsize=(7, 4))
plt.loglog(sizes, p_by_n, 'o-', color='C3')
plt.axhline(0.05, color='k', ls='--', label='p = 0.05')
plt.xlabel('stars per sample'); plt.ylabel('p-value')
plt.title(f'A {shift} dex shift nobody would care about, vs sample size')
plt.legend(); plt.show()

A shift far smaller than the measurement errors becomes "significant" once the sample is big enough - the p-value alone can't tell you the effect is real or that it matters.

In [ ]:
# presenter-only setup for the slider on the next slide (skip cell: executed, not shown)
from ipywidgets import interact, FloatSlider

def explore_cut(half_width):
    center = 0.2
    poor = d[d['alphFe'] < center - half_width]
    rich = d[d['alphFe'] > center + half_width]
    ks_w, p_w = stats.ks_2samp(poor['FeH'], rich['FeH'])
    plt.figure(figsize=(6, 4))
    plt.hist(poor['FeH'], bins=60, density=True, alpha=0.5, color='C0', label=f'alpha-poor (n={len(poor)})')
    plt.hist(rich['FeH'], bins=60, density=True, alpha=0.5, color='C1', label=f'alpha-rich (n={len(rich)})')
    plt.xlabel('[Fe/H]'); plt.ylabel('density'); plt.legend()
    plt.title(f'half-width={half_width:.2f}: D={ks_w:.3f}, p={p_w:.1e}')
    plt.show()

## Does the cut matter?

The 0.1/0.3 split was a choice, as I said a few slides ago. Drag the slider below to shrink the gap half-width toward zero and watch $D$ and $p$ in the plot title separately - they don't behave the same way, and that difference is the whole point of the exercise.


In [ ]:
interact(explore_cut, half_width=FloatSlider(value=0.10, min=0.0, max=0.15, step=0.01, description='gap half-width'));

Push the gap all the way to zero - alpha-poor and alpha-rich touching, nothing thrown away - and $p$ is still indistinguishable from zero, while $D$ drops into the low 0.4s. The effect got substantially smaller and the p-value had no way to report it, which is the large-sample trap again, this time on a knob you're turning yourself.


## The third trap: many tests

You just watched me slide one cut back and forth. Suppose instead we tried 20 different [Fe/H] and [alpha/Fe] binnings, looking for a significant difference somewhere.

Imagine those 20 comparisons where, in every single one, the two samples really are drawn from the same population - the null is true every time. Here is the fact that makes this bite: when $H_0$ is true, the p-value is uniformly distributed between 0 and 1, i.e. it is equally likely to land anywhere. So at a $p<0.05$ threshold about 1 test in 20 comes back "significant" even when every population was drawn from the same distribution.


## The multiple-comparisons problem

This is the **multiple-comparisons problem**, and it's what shows up when we let an AI assistant try a few different cuts and report what's significant without telling it to correct for how many cuts it tried.

Rule of thumb for this course: if you ran more than one test, say so, and say how many. The simplest correction, Bonferroni, just demands $p < 0.05/20$ instead of $p < 0.05$.

Let's watch that die-roll happen in a simulation first, then see how it played out for real at CERN and in a dead fish.


## Twenty true-null tests, 5,000 times over

We don't even need the stars for this. The fact from two slides ago says a true-null p-value is just a uniform random number between 0 and 1. So drawing 20 random numbers **is** running 20 tests where nothing is there, and every number below 0.05 is a false positive.

Do that once and you get a count between 0 and 20. Do it 5,000 times and you get the distribution of how many false positives a night of 20 tests hands you.

In [ ]:
rng2 = np.random.default_rng(20260827)
n_trials, n_tests = 5000, 20
# simulate 5000 "nights": each night, run 20 tests where the null is TRUE
false_positives = (rng2.uniform(0, 1, size=(n_trials, n_tests)) < 0.05).sum(axis=1)

plt.hist(false_positives, bins=np.arange(-0.5, 8.5, 1), rwidth=0.8)
plt.xlabel('number of "significant" (p<0.05) results out of 20 true-null tests')
plt.ylabel('count over 5000 simulated repeats')
plt.title(f"mean = {false_positives.mean():.2f} false positives per 20 tests, purely by chance")
plt.show()

About one false positive per 20 tests, on average, out of nothing at all. Now the real thing: hundreds of tests at once, on the most expensive instrument ever built.

## Background: the LHC and the two-photon bump hunt

The Large Hadron Collider at CERN, outside Geneva, smashes protons together at 13 TeV. ATLAS and CMS are the two big general-purpose detectors on the ring: separate hardware, separate collaborations of a few thousand people each, deliberately built to check one another.

In 2012 both found the Higgs boson at 125 GeV. One of the cleanest ways to see it: the Higgs sometimes decays into two photons. Measure both photons, compute the mass of the parent that would have produced them (the **diphoton invariant mass**), and histogram it over millions of collisions. Random photon pairs make a smooth, falling background. A new particle makes a bump on top of it at its own mass.


## Hundreds of tests in one plot

After the Higgs, everyone kept scanning that same histogram for heavier bumps. Every mass value tested is a separate hypothesis test - the plot below is hundreds of them at once. The *local* p-value is the test at one mass; the *global* significance is what's left after you admit you looked everywhere.

<img src="images/atlas_diphoton_local_pvalue.png" alt="ATLAS local p-value against diphoton mass from 200 to 2500 GeV, showing dozens of downward spikes past one and two sigma from noise alone, with an inset where the 2015 curve dives past three sigma near 750 GeV and the 2016 curve stays flat" width="960" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">ATLAS Collaboration, ATLAS-CONF-2016-059 Fig. 6(a), CC BY 4.0. Every spike is a test. Most are noise.</div>

## December 2015: a bump at 750 GeV

ATLAS and CMS both saw an excess in the diphoton mass spectrum: more photon pairs near 750 GeV than the smooth background predicts, six times the Higgs mass. Local significance 3.4 sigma - at that one mass. It drew several hundred theory papers within months, each proposing a new particle to explain it.

<img src="images/atlas_diphoton_2015.png" alt="ATLAS diphoton invariant mass spectrum from 3.2 inverse femtobarns of 2015 data with the background-only fit and a residual panel showing a positive excursion near 730 GeV" width="680" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">ATLAS-CONF-2016-059 Fig. 3(a), CC BY 4.0</div>

ATLAS's own 2015 note put the *global* significance at "about 2 standard deviations", once you account for having looked everywhere along that axis.

"N sigma" is just a p-value quoted as the equivalent Gaussian tail: 3 sigma is about $p = 0.001$, and particle physics does not say "discovery" below 5 sigma, $p \approx 3 \times 10^{-7}$. "Global" is exactly the Bonferroni move from a few slides ago, applied to the hundreds of masses tested.


## August 2016: four times the data

The LHC ran again in 2016 and recorded four times as many collisions (12.2 inverse femtobarns against 3.2 - the unit particle physicists use for how many collisions were recorded). If the particle were real, the bump should grow. Same plot, same binning, same fit:

<img src="images/atlas_diphoton_2016.png" alt="The same ATLAS diphoton mass spectrum with 12.2 inverse femtobarns of 2016 data, sitting flat on the background fit with no structure at 750 GeV" width="680" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">ATLAS-CONF-2016-059 Fig. 3(b), CC BY 4.0. Same binning, same fit, no bump.</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**"No significant excess... is observed in the 2016 data. The global significance is estimated to be less than one standard deviation."**

</div>


## And CMS saw it independently

CMS sits on the opposite side of the ring from ATLAS: a second detector, a different collaboration, and its own analysis, and it saw the same bump in the same place. These plots show the p-value at every mass tested, so a dip is a bump in the histogram. Left, CMS in 2015: a single deep spike at 750 GeV past 3 sigma. Right, after adding 2016 data: the blue 2015 curve still dives, the red 2016 curve does not dip at all.

<table style="border:none; margin:0 auto;"><tr style="border:none;">
<td style="border:none; padding:4px;"><img src="images/cms_diphoton_pvalue_2015.png" alt="CMS background-only p-value against resonance mass for 2015 data, with one deep spike at 750 GeV reaching past three sigma and many shallower dips elsewhere" width="450" style="display:block;margin:0 auto;"></td>
<td style="border:none; padding:4px;"><img src="images/cms_diphoton_pvalue_2016.png" alt="The same CMS p-value plot for 2015 plus 2016 data, with an inset showing the 2015 curve dipping at 750 GeV while the 2016 curve stays flat near p of 0.5" width="450" style="display:block;margin:0 auto;"></td>
</tr></table>

<div style="font-size:0.8em; text-align:center; color:#666;">CMS Collaboration, EXO-16-018 Fig. 4 and EXO-16-027 Fig. 5-a. © CERN for the benefit of the CMS Collaboration.</div>

Independent confirmation is a real and powerful check. It is not the same thing as being right, and two experiments looking in the same place with the same method can be fooled together.


## The dead salmon

fMRI maps brain activity by measuring blood-oxygen changes in each of roughly 130,000 little volume elements (voxels). The standard experiment: a person lies in the scanner and is shown photographs of people in social situations, asked to judge what emotion each one is feeling, and every voxel gets its own statistical test - active during the pictures or not.

Bennett and colleagues were testing a scanner protocol and needed a subject. They bought an Atlantic salmon at a market, put it in the scanner, ran the exact task above, and analysed it the way many papers of the day did: one test per voxel, no correction for having run 130,000 of them. The salmon was dead.


## What the salmon "saw"

<img src="images/salmon_fmri_uncorrected.png" alt="Two greyscale MRI slices of a whole Atlantic salmon with a red to white t-value colour scale, showing small bright clusters of apparently active voxels in the brain cavity and spinal column" width="900" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">Bennett, Baird, Miller &amp; Wolford, J. Serendipitous Unexpected Results 1(1), Fig. 1</div>

At $t > 3.15$ ($t$ is each voxel's signal-to-noise; 3.15 is where $p$ drops below 0.001), uncorrected, an 81 mm$^3$ cluster of "active" voxels shows up in the salmon's brain cavity. Correct for multiple comparisons and **zero** voxels survive, even at a relaxed 0.25 - whether you control the family-wise error rate (FWER: the chance of even one false positive across all 130,000 tests; Bonferroni is one way) or the false discovery rate (FDR: the fraction of your detections that are false). A typical fMRI volume is around 130,000 voxels, so at $p < 0.001$ uncorrected you expect roughly 130 false positives before you start.


## Checking the Gaussian assumption

That covers the three traps I promised you. The last thing I want to check is an assumption sitting underneath all of them: every least-squares fit from Tuesday, and every one in Lab 01, assumes Gaussian residuals. A QQ-plot is the fast visual check, the same one from the start of Day 3.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# left: the data itself, with the Gaussian we are implicitly assuming drawn over it
feh = alpha_poor['FeH']
mu, sd = feh.mean(), feh.std()
axes[0].hist(feh, bins=80, density=True, alpha=0.6, label='alpha-poor [Fe/H]')
xg = np.linspace(feh.min(), feh.max(), 400)
axes[0].plot(xg, stats.norm.pdf(xg, mu, sd), 'r-', lw=2,
             label=f'Gaussian, mean={mu:.2f}, sd={sd:.2f}')
axes[0].set_xlabel('[Fe/H]'); axes[0].set_ylabel('density'); axes[0].legend(fontsize=8)
axes[0].set_title('The data, with the Gaussian we assume')

# right: the same comparison, as a QQ-plot
stats.probplot(feh, dist='norm', plot=axes[1])
axes[1].set_title('The same comparison as a QQ-plot')
plt.tight_layout()
plt.show()

print(f"skew = {stats.skew(feh):+.2f}   (0 for a Gaussian; negative = longer tail to the left)")


## The cost of assuming Gaussian

Here's what that assumption costs on the alpha-poor sample we've been using all class.


In [ ]:
mean_feh, std_feh = alpha_poor['FeH'].mean(), alpha_poor['FeH'].std()
median_feh = np.median(alpha_poor['FeH'])
mad_sigma_feh = 1.4826 * np.median(np.abs(alpha_poor['FeH'] - median_feh))
print(f"assume Gaussian:  mean = {mean_feh:.2f}, std = {std_feh:.2f}")
print(f"robust instead:   median = {median_feh:.2f}, MAD-sigma = {mad_sigma_feh:.2f}")


The mean sits about 0.27 dex away from the median, pulled toward the metal-poor tail the same way the Cauchy demo pulled it on Tuesday. The naive standard deviation comes out nearly twice the robust MAD-sigma. Report "mean $\pm$ std" for a skewed sample without checking Gaussianity first, and you're describing the tail.


The QQ-plot and these numbers are saying the same thing. [Fe/H] is not Gaussian - the metal-poor tail peels away below the line, with a measured skew of about $-1.4$ - so the median and IQR from Day 2 are the right summaries for this sample.

In general, if the points follow the line then Gaussian is a fine working assumption. Where they peel off, usually in the tails, is where least-squares and "3-sigma means 99.7%" stop being true.


<img src="images/ceers_field_mosaic.png" alt="The CEERS Epoch 1 JWST NIRCam colour mosaic of the Extended Groth Strip, a long strip of sky crowded with galaxies of every colour and shape" width="1150" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">NASA/STScI/CEERS/TACC/S. Finkelstein/M. Bagley/Z. Levay &mdash; 690 frames, ten NIRCam pointings</div>


## Back to small samples: CEERS-93316

Our 8-star KS test couldn't decide. Here is the other way too little data fails you, out of the actual literature - it decides, with confidence, and goes to print wrong.

CEERS-93316 was reported in 2023 (Donnan et al.) as one of the most distant galaxies ever seen, a photometric redshift of $z \approx 16.4$, from seven broadband photometry points. CEERS was one of JWST's first deep-field programmes, in 2022; NIRCam is the telescope's near-infrared camera.


## Two ways to get a redshift

A spectrum shows you emission lines at measured wavelengths, and the redshift falls out directly. Photometry gives you one brightness per filter, a handful of numbers, and you fit galaxy templates to them: a **photometric redshift**. The high-$z$ argument rides on the **Lyman break**: neutral hydrogen along the line of sight absorbs essentially everything shortward of 1216 angstroms in the galaxy's rest frame, so at $z \approx 16$ the galaxy vanishes from every filter bluer than about 2 microns. That is what the two blank cutouts are.

<img src="images/ceers93316_stamps_sed.png" alt="Seven JWST NIRCam cutouts of CEERS-93316, blank in the two bluest filters and clearly detected in the redder ones, above the fitted spectral energy distribution through seven photometric points with an inset redshift posterior spiking at 16.4" width="1000" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">Donnan et al. 2023, MNRAS 518, 6011 (arXiv:2207.12356), CC BY 4.0</div>


## A tight posterior, and wrong

<img src="images/ceers93316_photoz_pdf.png" alt="Photometric redshift posterior distributions for the CEERS high-redshift candidates, with CEERS-93316 a narrow confident spike near redshift 16.4 and the true spectroscopic redshift of 4.91 lying far outside it" width="820" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">Arrabal Haro et al. 2023, Nature 622, 707, Fig. 1, CC BY 4.0</div>

The posterior - the probability distribution over redshift that the fit hands back, whose width is the quoted error bar - is narrow. It is also nowhere near the right answer. A tight error bar is a statement about the model you fitted, not a promise about the world.


## What the spectrum said

<img src="images/ceers93316_nirspec_spectrum.png" alt="JWST NIRSpec prism spectrum of CEERS-93316 in two and one dimensions, with H-beta, doubly ionised oxygen and H-alpha emission lines labelled, giving a redshift of 4.912" width="1000" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">Arrabal Haro et al. 2023, Nature 622, 707, Fig. 3, CC BY 4.0</div>

Two emission lines placed it at $z = 4.91$, a dusty, lower-redshift starburst. The 4000-angstrom break is a weaker, far more common feature of older stellar populations. Redshift it to $z \approx 5$ and it lands near 2.4 microns, close to where a Lyman break at $z \approx 16$ would sit, and strong [OIII] and H-alpha emission lines pumped the redder filters enough to fill in the rest. Seven photometric points could not tell the two apart.

This is not the 8-star problem. The failure is not too few objects but a degeneracy: two very different galaxies producing the same seven numbers. More of the same photometry would not have fixed that - a different and more informative measurement did (NIRSpec is JWST's spectrograph).


## Background: the polarised microwave sky

The cosmic microwave background is light released when the universe was about 380,000 years old, redshifted today into microwaves. It is faintly polarised, and a polarisation pattern on the sky splits into two kinds: a curl-free **E-mode** and a curl-like **B-mode**.

Inflation, the proposed burst of exponential expansion in the first fraction of a second, predicts primordial gravitational waves, and those are the only *primordial* source of B-modes. Their amplitude is the tensor-to-scalar ratio $r$. Measure $r > 0$ and you have direct evidence for inflation.


## Three ways to make a B-mode

Inflation is one source. Two other things make B-modes. Gravitational lensing of the CMB by everything between us and it turns some E-mode into B-mode - that is the "lensed $\Lambda$CDM" curve, the floor you expect with $r = 0$. And polarised emission from dust grains in our own Galaxy, a **foreground** sitting on top of everything.

BICEP2 was a small telescope at the South Pole observing at one frequency, 150 GHz, where the CMB is bright. At one frequency you cannot tell dust from CMB. Planck, the ESA satellite that mapped the whole sky in nine frequency bands, has a 353 GHz channel where dust dominates - effectively a dust map.

## The other one: BICEP2 and the dust

In March 2014 BICEP2 announced a detection of primordial gravitational waves from inflation, at high significance, and it was front-page news. Read as $r \approx 0.2$. Black points here are the raw B-mode signal, sitting well above the lensing-only prediction.

<img src="images/bicep2_planck_dust.png" alt="B-mode power spectrum with black points from BICEP2 and Keck sitting above the lensed lambda-CDM curve, and blue points after subtracting the dust contribution measured against Planck falling onto that curve, with the likelihood for r below peaking near zero" width="560" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">BICEP2/Keck and Planck Collaborations, Phys. Rev. Lett. 114, 101301 (2015), Fig. 12</div>

The blue points are the same data after subtracting the polarised galactic dust measured by cross-correlating with Planck at 353 GHz. They land on the lensing curve, and the joint analysis left only an upper limit, $r < 0.12$. The signal was foreground. As with CEERS-93316, what settled it was not more of the same measurement but a different and more informative one.


## What "thick disk" means

That one was somebody else's data. Here is the same kind of problem sitting underneath today's demo.

The thick disk started as a geometric claim (Gilmore & Reid 1983): count stars against height above the Galactic mid-plane and you need a second, thicker exponential to fit the counts. The **scale height** is that exponential's e-folding thickness. A **mono-abundance population** is every star in one small box of the [Fe/H]-[alpha/Fe] plane. If there really are two disks, sort the boxes by scale height and you should get two clumps, thin and thick.


## Bovy, Rix & Hogg (2012)

Bovy, Rix & Hogg (2012, *ApJ*, 751, 131; arXiv:1111.6585) ran mono-abundance sub-populations from this same SEGUE survey through a spatial-structure analysis and found "no hint of a thin-thick disk bi-modality" - instead, a continuous and monotonic distribution of disk thicknesses.

<img src="images/bovy2012_scaleheight_continuum.png" alt="Surface mass density against vertical scale height for mono-abundance populations, coloured by alpha to iron ratio, with points filling 200 to 1100 parsecs smoothly and no gap or second clump" width="760" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">Bovy, Rix &amp; Hogg 2012, ApJ 751, 131, Fig. 2. If there were a distinct thick disk, there would be a gap here.</div>


## Counts, versus mass

Look back at our very first plot today. That 2D histogram is a *number count*: one dot per star that made it into the catalogue. The **selection function** is the probability that a star of a given brightness, colour and position survived the ten cuts from earlier. Divide each star's contribution by that probability, weight by the stellar mass it stands for, and you recover what is out there rather than what was easy to observe. Do that, and the second clump goes away.

<table style="border:none; margin:0 auto;"><tr style="border:none;">
<td style="border:none; padding:4px;"><img src="images/bovy2012_raw_counts.png" alt="Alpha to iron versus iron abundance for the raw SEGUE sample counts, whose marginal alpha histogram shows a peak near 0.1 and a second bump near 0.4" width="430" style="display:block;margin:0 auto;"></td>
<td style="border:none; padding:4px;"><img src="images/bovy2012_mass_weighted.png" alt="The same plane weighted by stellar mass, whose marginal alpha histogram is a single peak near 0.1 falling away monotonically" width="430" style="display:block;margin:0 auto;"></td>
</tr></table>

<div style="font-size:0.8em; text-align:center; color:#666;">Bovy, Rix &amp; Hogg 2012, ApJ 751, 131, Fig. 1. Watch the right-hand marginal histogram in each panel.</div>

I've used a two-population framing all class - thin disk vs. thick disk - and simplified an open research question into a clean classroom split. Same data, a different weighting, and the two-populations story gets much harder to defend.


## Open problems in this dataset

You have the pieces these need: the selection caveats, the KS test, and the QQ-plot.

* Correct the selection function. Forward-model the catalog's ten selection cuts through a Milky Way stellar population model and ask how the [Fe/H] and [alpha/Fe] distribution you'd infer changes once selection is accounted for - a well-posed and tractable project.
* Cross-check against modern astrometry. This is SEGUE data from SDSS Data Release 9 (2012). Gaia, ESA's astrometry satellite launched in 2013, now gives parallaxes and proper motions for 1.8 billion stars, these included. Do the thin and thick disk populations hold up once you add real distances and orbits, instead of inferring membership from chemistry alone?


## Open problems, continued

* Chemistry against kinematics. This catalog already has radial velocities and proper motions in it, unused in today's demo. Do chemically-defined populations and kinematically-defined ones agree about which stars belong to which disk component? They don't always, and that disagreement is an active research question.
* Age-abundance relations. This catalog alone has no ages, but cross-matching against surveys that do, like asteroseismology (ages from the oscillation frequencies of a star's surface, from Kepler and TESS light curves) or isochrone fitting (placing a star on model tracks of luminosity and temperature against age), lets you test whether alpha-rich always means old, and nobody has a settled answer to that right now.

<img src="images/gaia_allsky.jpg" alt="The Gaia all-sky map in colour, showing the Milky Way disk, dark dust lanes and both Magellanic Clouds" width="1000" style="display:block;margin:0 auto;">

<div style="font-size:0.8em; text-align:center; color:#666;">ESA/Gaia/DPAC, CC BY-SA 3.0 IGO &mdash; 1.8 billion stars, and this catalogue has 252,871</div>


## Wrapping up

The answer: the two [Fe/H] distributions do differ. $D = 0.61$, and $p$ underflowed. But it's $D$ that makes that worth saying - at 130,000 stars the p-value was going to be tiny either way.

It is also a weaker claim than "there are two disk populations". A KS test says two *samples* differ. It does not say the parent population is bimodal, and that is the part Bovy, Rix & Hogg dispute.

The three traps, in the concrete form you saw them today:

* 8 stars missed a 0.75 dex separation more than half the time
* 130,000 stars gave $p \approx 0$ wherever we put the cut
* 20 tests hand you one "significant" result out of nothing at all

Report the effect size, and how many tests you ran, alongside $p$. An AI assistant will add neither unprompted.


## Before you go

Underneath all of it is the Day 2 question: how do we check that our data, our fit and our test are telling us the truth?

* four robust fits from Tuesday - clip, MAD, mixture, Theil-Sen
* the bootstrap, for an error bar on almost anything
* the QQ-plot, for whether Gaussian was ever a fair assumption
* today, the KS test and its three traps

**Lab 01** is posted, due Wed Sep 9 at Noon. Fork then PR as always. You build the k-sigma clipping and the mixture model yourself, on your own dataset.

Nothing from today is collected - Lab 01 is the deliverable.
